# 14. Network domain models and parsing

**Level:** Intermediate

**Learning outcome:** Model addresses, route membership, VLAN intent, and explicit parsing boundaries without live I/O.

**Prerequisites:** 03-04 and 11: validation, composition, and records.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](13_Context_Managers_and_Resource_Lifetime.ipynb) | [Next notebook](15_Domain_Exceptions_Retries_and_Resilience.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Intermediate

**Learning goal:** Represent addresses, routes, VLAN sets, and simplified observations with explicit assumptions.

**A model is not a network protocol implementation.** The examples use Python's ipaddress
objects for address arithmetic rather than splitting dotted strings manually. Membership
answers 'is this address in this network?' and not 'is this host reachable?'. Likewise, a
selected route here is a teaching model, not a full vendor RIB/FIB calculation.

**IPv4 and IPv6.** Use ip_address for either address family, ip_network for a prefix, and
ip_interface for an address with its prefix length. Strict network construction rejects
host bits; strict=False intentionally normalizes to a network boundary. Do not silently mix
address families. IPv6 does not use IPv4 broadcast semantics. Avoid enumerating every address
of a large prefix; a /64 is not a list you should materialize.

**Parsing is a trust boundary.** A parser should accept one documented grammar, normalize
deliberately, and report malformed data rather than convert it into an apparently healthy
status. These lesson parsers accept tiny synthetic formats. They are not robust vendor CLI
parsers and make no claim to handle localized, truncated, paginated, or version-changing output.

**Useful collections.** Sets model allowed VLAN membership and difference; dictionaries model
exact lookup; a sequence plus an explicit key can implement longest-prefix selection for a
small lab. A production routing table may require a prefix tree and more detailed policies.
Keep numeric values numeric when comparing costs or counters; lexical string order is not
numeric order.

**Operational interpretation.** An admin-up interface can still be operationally down. A
matching ACL prefix in this simplified model is not a complete ACL evaluation: protocols,
ports, order, direction, VRF, and platform behavior are intentionally outside that exercise.

**Pause and explain:** Distinguish address membership, route selection, and actual reachability.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 041: Prefix containment as a domain operation

**Core concept:** Wrap an ipaddress network behind a meaningful method.

**Network-engineering purpose:** A policy can ask whether a management address belongs to an approved lab subnet.

### Read the implementation in this order
1. Normalize the configured prefix with ip_network.
2. Parse the candidate with ip_address.
3. Use the membership operation.
4. Check one member and one explicitly unrelated address.

**Common mistake / boundary:** Containment does not test ping, routing, ACL permission, or service availability.

**Practice after the ten worked scenarios:** Reject an address-family mismatch explicitly rather than treating it as an ordinary miss.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0401 - Prefix containment as a domain operation: branch scenario

**Scenario 01/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.0/28'` |
| `ip` | `'192.0.2.3'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.3 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [1]:
# Example 0401: Prefix containment as a domain operation - branch
case = {'prefix': '192.0.2.0/28', 'ip': '192.0.2.3'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.3 True


### Example 0402 - Prefix containment as a domain operation: campus scenario

**Scenario 02/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.16/28'` |
| `ip` | `'192.0.2.19'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.19 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [2]:
# Example 0402: Prefix containment as a domain operation - campus
case = {'prefix': '192.0.2.16/28', 'ip': '192.0.2.19'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.19 True


### Example 0403 - Prefix containment as a domain operation: dc scenario

**Scenario 03/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.32/28'` |
| `ip` | `'192.0.2.35'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.35 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [3]:
# Example 0403: Prefix containment as a domain operation - dc
case = {'prefix': '192.0.2.32/28', 'ip': '192.0.2.35'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.35 True


### Example 0404 - Prefix containment as a domain operation: wan scenario

**Scenario 04/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.48/28'` |
| `ip` | `'192.0.2.51'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.51 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [4]:
# Example 0404: Prefix containment as a domain operation - wan
case = {'prefix': '192.0.2.48/28', 'ip': '192.0.2.51'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.51 True


### Example 0405 - Prefix containment as a domain operation: lab scenario

**Scenario 05/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.64/28'` |
| `ip` | `'192.0.2.67'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.67 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [5]:
# Example 0405: Prefix containment as a domain operation - lab
case = {'prefix': '192.0.2.64/28', 'ip': '192.0.2.67'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.67 True


### Example 0406 - Prefix containment as a domain operation: wireless scenario

**Scenario 06/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.80/28'` |
| `ip` | `'192.0.2.83'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.83 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [6]:
# Example 0406: Prefix containment as a domain operation - wireless
case = {'prefix': '192.0.2.80/28', 'ip': '192.0.2.83'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.83 True


### Example 0407 - Prefix containment as a domain operation: remote scenario

**Scenario 07/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.96/28'` |
| `ip` | `'192.0.2.99'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.99 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [7]:
# Example 0407: Prefix containment as a domain operation - remote
case = {'prefix': '192.0.2.96/28', 'ip': '192.0.2.99'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.99 True


### Example 0408 - Prefix containment as a domain operation: core scenario

**Scenario 08/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.112/28'` |
| `ip` | `'192.0.2.115'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.115 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [8]:
# Example 0408: Prefix containment as a domain operation - core
case = {'prefix': '192.0.2.112/28', 'ip': '192.0.2.115'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.115 True


### Example 0409 - Prefix containment as a domain operation: mgmt scenario

**Scenario 09/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.128/28'` |
| `ip` | `'192.0.2.131'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.131 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [9]:
# Example 0409: Prefix containment as a domain operation - mgmt
case = {'prefix': '192.0.2.128/28', 'ip': '192.0.2.131'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.131 True


### Example 0410 - Prefix containment as a domain operation: backup scenario

**Scenario 10/10; pattern 041; Intermediate.** This is a self-contained offline exercise. A policy can ask whether a management address belongs to an approved lab subnet.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.144/28'` |
| `ip` | `'192.0.2.147'` |

**Logic to trace:** Normalize the configured prefix with ip_network. Check one member and one explicitly unrelated address. Follow the class-specific steps under Pattern 041 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.147 True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Containment does not test ping, routing, ACL permission, or service availability.

In [10]:
# Example 0410: Prefix containment as a domain operation - backup
case = {'prefix': '192.0.2.144/28', 'ip': '192.0.2.147'}

from ipaddress import ip_address, ip_network

class PrefixPolicy:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def contains(self, address):
        return ip_address(address) in self.network

policy = PrefixPolicy(case["prefix"])
assert policy.contains(case["ip"])
assert not policy.contains("203.0.113.250")
print(case["ip"], policy.contains(case["ip"]))


192.0.2.147 True


## Pattern 042: Longest-prefix route lookup

**Core concept:** A route table can select the matching destination with the greatest prefix length.

**Network-engineering purpose:** An engineer can reason about a specific route winning over a default in a small offline model.

### Read the implementation in this order
1. Parse route prefixes once in the constructor.
2. Filter routes containing the destination.
3. Choose max prefixlen or raise when none match.
4. Check the scenario's specific route beats default.

**Common mistake / boundary:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

**Practice after the ten worked scenarios:** Add an equal-prefix tie policy and document the result.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0411 - Longest-prefix route lookup: branch scenario

**Scenario 01/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.0/28'` |
| `ip` | `'192.0.2.3'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [11]:
# Example 0411: Longest-prefix route lookup - branch
case = {'prefix': '192.0.2.0/28', 'ip': '192.0.2.3'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0412 - Longest-prefix route lookup: campus scenario

**Scenario 02/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.16/28'` |
| `ip` | `'192.0.2.19'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [12]:
# Example 0412: Longest-prefix route lookup - campus
case = {'prefix': '192.0.2.16/28', 'ip': '192.0.2.19'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0413 - Longest-prefix route lookup: dc scenario

**Scenario 03/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.32/28'` |
| `ip` | `'192.0.2.35'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [13]:
# Example 0413: Longest-prefix route lookup - dc
case = {'prefix': '192.0.2.32/28', 'ip': '192.0.2.35'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0414 - Longest-prefix route lookup: wan scenario

**Scenario 04/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.48/28'` |
| `ip` | `'192.0.2.51'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [14]:
# Example 0414: Longest-prefix route lookup - wan
case = {'prefix': '192.0.2.48/28', 'ip': '192.0.2.51'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0415 - Longest-prefix route lookup: lab scenario

**Scenario 05/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.64/28'` |
| `ip` | `'192.0.2.67'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [15]:
# Example 0415: Longest-prefix route lookup - lab
case = {'prefix': '192.0.2.64/28', 'ip': '192.0.2.67'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0416 - Longest-prefix route lookup: wireless scenario

**Scenario 06/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.80/28'` |
| `ip` | `'192.0.2.83'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [16]:
# Example 0416: Longest-prefix route lookup - wireless
case = {'prefix': '192.0.2.80/28', 'ip': '192.0.2.83'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0417 - Longest-prefix route lookup: remote scenario

**Scenario 07/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.96/28'` |
| `ip` | `'192.0.2.99'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [17]:
# Example 0417: Longest-prefix route lookup - remote
case = {'prefix': '192.0.2.96/28', 'ip': '192.0.2.99'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0418 - Longest-prefix route lookup: core scenario

**Scenario 08/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.112/28'` |
| `ip` | `'192.0.2.115'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [18]:
# Example 0418: Longest-prefix route lookup - core
case = {'prefix': '192.0.2.112/28', 'ip': '192.0.2.115'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0419 - Longest-prefix route lookup: mgmt scenario

**Scenario 09/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.128/28'` |
| `ip` | `'192.0.2.131'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [19]:
# Example 0419: Longest-prefix route lookup - mgmt
case = {'prefix': '192.0.2.128/28', 'ip': '192.0.2.131'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


### Example 0420 - Longest-prefix route lookup: backup scenario

**Scenario 10/10; pattern 042; Intermediate.** This is a self-contained offline exercise. An engineer can reason about a specific route winning over a default in a small offline model.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.144/28'` |
| `ip` | `'192.0.2.147'` |

**Logic to trace:** Parse route prefixes once in the constructor. Check the scenario's specific route beats default. Follow the class-specific steps under Pattern 042 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
specific
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This model omits route installation, administrative preference, metric, ECMP, and forwarding verification.

In [20]:
# Example 0420: Longest-prefix route lookup - backup
case = {'prefix': '192.0.2.144/28', 'ip': '192.0.2.147'}

from ipaddress import ip_address, ip_network

class RouteTable:
    def __init__(self, records):
        self.routes = [(ip_network(prefix), hop) for prefix, hop in records]

    def lookup(self, address):
        destination = ip_address(address)
        matches = [item for item in self.routes if destination in item[0]]
        if not matches:
            raise LookupError("no matching route")
        return max(matches, key=lambda item: item[0].prefixlen)[1]

table = RouteTable([("0.0.0.0/0", "default"), (case["prefix"], "specific")])
assert table.lookup(case["ip"]) == "specific"
print(table.lookup(case["ip"]))


specific


## Pattern 043: VLAN set reconciliation

**Core concept:** Set difference expresses desired additions and removals without duplicate commands.

**Network-engineering purpose:** A trunk intent model can compare unordered memberships rather than raw CLI text.

### Read the implementation in this order
1. Store allowed VLANs as a set.
2. Compute desired minus observed for additions.
3. Compute observed minus desired for removals.
4. Sort the report for deterministic output.

**Common mistake / boundary:** A set ignores ordering; use it only where ordering has no semantic meaning.

**Practice after the ten worked scenarios:** Add policy validation before producing a plan and reject reserved IDs for your chosen platform.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0421 - VLAN set reconciliation: branch scenario

**Scenario 01/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `1` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [2] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [21]:
# Example 0421: VLAN set reconciliation - branch
case = {'vlan': 1}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [2] remove: [4094]


### Example 0422 - VLAN set reconciliation: campus scenario

**Scenario 02/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `10` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [11] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [22]:
# Example 0422: VLAN set reconciliation - campus
case = {'vlan': 10}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [11] remove: [4094]


### Example 0423 - VLAN set reconciliation: dc scenario

**Scenario 03/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `100` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [101] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [23]:
# Example 0423: VLAN set reconciliation - dc
case = {'vlan': 100}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [101] remove: [4094]


### Example 0424 - VLAN set reconciliation: wan scenario

**Scenario 04/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `200` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [201] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [24]:
# Example 0424: VLAN set reconciliation - wan
case = {'vlan': 200}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [201] remove: [4094]


### Example 0425 - VLAN set reconciliation: lab scenario

**Scenario 05/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `999` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [1000] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [25]:
# Example 0425: VLAN set reconciliation - lab
case = {'vlan': 999}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [1000] remove: [4094]


### Example 0426 - VLAN set reconciliation: wireless scenario

**Scenario 06/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `1000` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [1001] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [26]:
# Example 0426: VLAN set reconciliation - wireless
case = {'vlan': 1000}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [1001] remove: [4094]


### Example 0427 - VLAN set reconciliation: remote scenario

**Scenario 07/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `2000` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [2001] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [27]:
# Example 0427: VLAN set reconciliation - remote
case = {'vlan': 2000}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [2001] remove: [4094]


### Example 0428 - VLAN set reconciliation: core scenario

**Scenario 08/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `3000` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [3001] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [28]:
# Example 0428: VLAN set reconciliation - core
case = {'vlan': 3000}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [3001] remove: [4094]


### Example 0429 - VLAN set reconciliation: mgmt scenario

**Scenario 09/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `4000` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [4001] remove: [4094]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [29]:
# Example 0429: VLAN set reconciliation - mgmt
case = {'vlan': 4000}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add: [4001] remove: [4094]


### Example 0430 - VLAN set reconciliation: backup scenario

**Scenario 10/10; pattern 043; Intermediate.** This is a self-contained offline exercise. A trunk intent model can compare unordered memberships rather than raw CLI text.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `4093` |

**Logic to trace:** Store allowed VLANs as a set. Sort the report for deterministic output. Follow the class-specific steps under Pattern 043 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
add: [] remove: []
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A set ignores ordering; use it only where ordering has no semantic meaning.

In [30]:
# Example 0430: VLAN set reconciliation - backup
case = {'vlan': 4093}

class VlanIntent:
    def __init__(self, allowed):
        self.allowed = set(allowed)

    def diff(self, observed):
        current = set(observed)
        return sorted(self.allowed - current), sorted(current - self.allowed)

intent = VlanIntent([case["vlan"], case["vlan"] + 1])
added, removed = intent.diff([case["vlan"], 4094])
if case["vlan"] + 1 == 4094:
    assert added == [] and removed == []
else:
    assert added == [case["vlan"] + 1] and removed == [4094]
print("add:", added, "remove:", removed)


add:

 [] remove: []


## Pattern 044: Parse a tiny interface observation

**Core concept:** A parser converts one declared external grammar into a validated object.

**Network-engineering purpose:** Telemetry consumers should receive structured status instead of reparsing strings everywhere.

### Read the implementation in this order
1. Require exactly a name and one status token.
2. Accept only up or down.
3. Build the object through a named constructor.
4. Check valid parsing and explicit rejection of malformed state.

**Common mistake / boundary:** Do not use this two-token grammar on arbitrary real-device output.

**Practice after the ten worked scenarios:** Add a test for missing tokens and a separate test for extra tokens.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0431 - Parse a tiny interface observation: branch scenario

**Scenario 01/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |
| `healthy` | `True` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet1/1', up=True)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [31]:
# Example 0431: Parse a tiny interface observation - branch
case = {'interface': 'Ethernet1/1', 'healthy': True}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected
Observation(interface='Ethernet1/1', up=True)


### Example 0432 - Parse a tiny interface observation: campus scenario

**Scenario 02/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |
| `healthy` | `False` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet2/1', up=False)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [32]:
# Example 0432: Parse a tiny interface observation - campus
case = {'interface': 'Ethernet2/1', 'healthy': False}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected
Observation(interface='Ethernet2/1', up=False)


### Example 0433 - Parse a tiny interface observation: dc scenario

**Scenario 03/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |
| `healthy` | `True` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet3/1', up=True)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [33]:
# Example 0433: Parse a tiny interface observation - dc
case = {'interface': 'Ethernet3/1', 'healthy': True}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected
Observation(interface='Ethernet3/1', up=True)


### Example 0434 - Parse a tiny interface observation: wan scenario

**Scenario 04/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |
| `healthy` | `True` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet4/1', up=True)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [34]:
# Example 0434: Parse a tiny interface observation - wan
case = {'interface': 'Ethernet4/1', 'healthy': True}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected
Observation(interface='Ethernet4/1', up=True)


### Example 0435 - Parse a tiny interface observation: lab scenario

**Scenario 05/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |
| `healthy` | `False` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet5/1', up=False)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [35]:
# Example 0435: Parse a tiny interface observation - lab
case = {'interface': 'Ethernet5/1', 'healthy': False}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected
Observation(interface='Ethernet5/1', up=False)


### Example 0436 - Parse a tiny interface observation: wireless scenario

**Scenario 06/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |
| `healthy` | `True` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet6/1', up=True)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [36]:
# Example 0436: Parse a tiny interface observation - wireless
case = {'interface': 'Ethernet6/1', 'healthy': True}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected
Observation(interface='Ethernet6/1', up=True)


### Example 0437 - Parse a tiny interface observation: remote scenario

**Scenario 07/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |
| `healthy` | `False` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet7/1', up=False)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [37]:
# Example 0437: Parse a tiny interface observation - remote
case = {'interface': 'Ethernet7/1', 'healthy': False}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected
Observation(interface='Ethernet7/1', up=False)


### Example 0438 - Parse a tiny interface observation: core scenario

**Scenario 08/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |
| `healthy` | `True` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet8/1', up=True)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [38]:
# Example 0438: Parse a tiny interface observation - core
case = {'interface': 'Ethernet8/1', 'healthy': True}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected


Observation(interface='Ethernet8/1', up=True)


### Example 0439 - Parse a tiny interface observation: mgmt scenario

**Scenario 09/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |
| `healthy` | `False` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet9/1', up=False)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [39]:
# Example 0439: Parse a tiny interface observation - mgmt
case = {'interface': 'Ethernet9/1', 'healthy': False}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected


Observation(interface='Ethernet9/1', up=False)


### Example 0440 - Parse a tiny interface observation: backup scenario

**Scenario 10/10; pattern 044; Intermediate.** This is a self-contained offline exercise. Telemetry consumers should receive structured status instead of reparsing strings everywhere.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |
| `healthy` | `True` |

**Logic to trace:** Require exactly a name and one status token. Check valid parsing and explicit rejection of malformed state. Follow the class-specific steps under Pattern 044 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
malformed observation rejected
Observation(interface='Ethernet10/1', up=True)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not use this two-token grammar on arbitrary real-device output.

In [40]:
# Example 0440: Parse a tiny interface observation - backup
case = {'interface': 'Ethernet10/1', 'healthy': True}

from dataclasses import dataclass

@dataclass
class Observation:
    interface: str
    up: bool

    @classmethod
    def parse(cls, text):
        fields = text.split()
        if len(fields) != 2 or fields[1] not in {"up", "down"}:
            raise ValueError("expected: interface up|down")
        return cls(fields[0], fields[1] == "up")

word = "up" if case["healthy"] else "down"
item = Observation.parse(f"{case['interface']} {word}")
assert item.up == case["healthy"]
try:
    Observation.parse("port unknown")
except ValueError:
    print("malformed observation rejected")
else:
    raise AssertionError("malformed observation accepted")
print(item)


malformed observation rejected
Observation(interface='Ethernet10/1', up=True)


## Pattern 045: Address family and canonical interface values

**Core concept:** ip_interface keeps a host address together with its prefix.

**Network-engineering purpose:** Canonical formatting avoids manual IPv6 string normalization.

### Read the implementation in this order
1. Parse an address/prefix pair.
2. Expose its canonical host and network strings.
3. Handle IPv4 and IPv6 through the same API.
4. Check the expected family from the scenario.

**Common mistake / boundary:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

**Practice after the ten worked scenarios:** Add a constructor invariant requiring the address family expected by the inventory record.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0441 - Address family and canonical interface values: branch scenario

**Scenario 01/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'192.0.2.3/28'` |
| `family` | `4` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.3 192.0.2.0/28 IPv4
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [41]:
# Example 0441: Address family and canonical interface values - branch
case = {'assignment': '192.0.2.3/28', 'family': 4}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


192.0.2.3

 192.0.2.0/28 IPv4


### Example 0442 - Address family and canonical interface values: campus scenario

**Scenario 02/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'2001:db8:2::3/64'` |
| `family` | `6` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
2001:db8:2::3 2001:db8:2::/64 IPv6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [42]:
# Example 0442: Address family and canonical interface values - campus
case = {'assignment': '2001:db8:2::3/64', 'family': 6}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


2001:db8:2::3 2001:db8:2::/64 IPv6


### Example 0443 - Address family and canonical interface values: dc scenario

**Scenario 03/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'192.0.2.35/28'` |
| `family` | `4` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.35 192.0.2.32/28 IPv4
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [43]:
# Example 0443: Address family and canonical interface values - dc
case = {'assignment': '192.0.2.35/28', 'family': 4}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


192.0.2.35 192.0.2.32/28 IPv4


### Example 0444 - Address family and canonical interface values: wan scenario

**Scenario 04/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'2001:db8:4::3/64'` |
| `family` | `6` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
2001:db8:4::3 2001:db8:4::/64 IPv6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [44]:
# Example 0444: Address family and canonical interface values - wan
case = {'assignment': '2001:db8:4::3/64', 'family': 6}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


2001:db8:4::3 2001:db8:4::/64 IPv6


### Example 0445 - Address family and canonical interface values: lab scenario

**Scenario 05/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'192.0.2.67/28'` |
| `family` | `4` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.67 192.0.2.64/28 IPv4
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [45]:
# Example 0445: Address family and canonical interface values - lab
case = {'assignment': '192.0.2.67/28', 'family': 4}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


192.0.2.67 192.0.2.64/28 IPv4


### Example 0446 - Address family and canonical interface values: wireless scenario

**Scenario 06/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'2001:db8:6::3/64'` |
| `family` | `6` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
2001:db8:6::3 2001:db8:6::/64 IPv6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [46]:
# Example 0446: Address family and canonical interface values - wireless
case = {'assignment': '2001:db8:6::3/64', 'family': 6}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


2001:db8:6::3 2001:db8:6::/64 IPv6


### Example 0447 - Address family and canonical interface values: remote scenario

**Scenario 07/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'192.0.2.99/28'` |
| `family` | `4` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.99 192.0.2.96/28 IPv4
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [47]:
# Example 0447: Address family and canonical interface values - remote
case = {'assignment': '192.0.2.99/28', 'family': 4}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


192.0.2.99 192.0.2.96/28 IPv4


### Example 0448 - Address family and canonical interface values: core scenario

**Scenario 08/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'2001:db8:8::3/64'` |
| `family` | `6` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
2001:db8:8::3 2001:db8:8::/64 IPv6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [48]:
# Example 0448: Address family and canonical interface values - core
case = {'assignment': '2001:db8:8::3/64', 'family': 6}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


2001:db8:8::3 2001:db8:8::/64 IPv6


### Example 0449 - Address family and canonical interface values: mgmt scenario

**Scenario 09/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'192.0.2.131/28'` |
| `family` | `4` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
192.0.2.131 192.0.2.128/28 IPv4
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [49]:
# Example 0449: Address family and canonical interface values - mgmt
case = {'assignment': '192.0.2.131/28', 'family': 4}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


192.0.2.131 192.0.2.128/28 IPv4


### Example 0450 - Address family and canonical interface values: backup scenario

**Scenario 10/10; pattern 045; Intermediate.** This is a self-contained offline exercise. Canonical formatting avoids manual IPv6 string normalization.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `assignment` | `'2001:db8:10::3/64'` |
| `family` | `6` |

**Logic to trace:** Parse an address/prefix pair. Check the expected family from the scenario. Follow the class-specific steps under Pattern 045 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
2001:db8:10::3 2001:db8:10::/64 IPv6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Formatting an address is not validation of reachability, assignment, or duplicate-address absence.

In [50]:
# Example 0450: Address family and canonical interface values - backup
case = {'assignment': '2001:db8:10::3/64', 'family': 6}

from ipaddress import ip_interface

class AddressAssignment:
    def __init__(self, address):
        self.value = ip_interface(address)

    def summary(self):
        return str(self.value.ip), str(self.value.network)

assignment = AddressAssignment(case["assignment"])
host, network = assignment.summary()
assert assignment.value.version == case["family"]
assert "/" in network and host == str(assignment.value.ip)
print(host, network, "IPv" + str(assignment.value.version))


2001:db8:10::3 2001:db8:10::/64 IPv6


## Check your understanding: Network domain models and parsing

**You should now be able to:** Model addresses, route membership, VLAN intent, and explicit parsing boundaries without live I/O.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.